# Rulkov map 0320 workflow

?? notebook ?? `example_maps_Q` ????????????? `twogroup_gram_0313.ipynb` ????????

- ??????? Rulkov map
- ??? `5000` ????????? `5000-7000` ??????? Koopman ??
- ??? PCA
- ????????? `cmap="vlag"`
- ????????score???????? K?SVD?positive contributions?EC/CE??????????


## 1. ?????

????????

- ?????????? notebook ????????????
- ?? Rulkov map ??????
- ???????????


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps
import scipy.linalg
import seaborn as sns
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
repo_root = next(path for path in [cwd, *cwd.parents] if (path / 'tools' / 'tools.py').exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

from exp.map.rulkov_map_tools import (
    ObservableConfig,
    RulkovSimulationConfig,
    WorkflowConfig,
    build_observables,
    build_state_matrix_from_population_data,
    channel_scores_from_singular_values,
    compute_entropy,
    compute_residual_covariance,
    fit_data_koopman_operator,
    format_equations,
    generate_two_population_neuron_data,
    get_positive_contributions,
    koopman_ce_total_score_from_kbar,
    liu2025_log_gamma_gis,
    plot_macro_series,
    plot_matrix_heatmap,
    plot_micro_macro_comparison,
    plot_neuron_analysis_combo,
    plot_positive_contributions,
    plot_rectangular_heatmap,
    plot_singular_value_comparison,
    save_workflow_results,
    set_plot_style,
    whiten_operator_matrix,
)

set_plot_style()
pd.set_option('display.max_columns', 24)
pd.set_option('display.width', 180)
np.set_printoptions(precision=4, suppress=True)

results_dir = repo_root / 'exp' / 'map' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
print('repo_root =', repo_root)
print('results_dir =', results_dir)
print('font.sans-serif =', plt.rcParams['font.sans-serif'])


## 2. ????

????????

- ?????????? Rulkov ??
- ???? observables ? `identity + quadratic`
- ????????? rank ????


In [ ]:
seed = 103
dt = 1.0
discrete_time = True
simulation_config = RulkovSimulationConfig(
    n_a=100,
    n_b=100,
    alpha_a=4.6,
    alpha_b=4.6,
    sigma_a=0.225,
    sigma_b=0.225,
    mu=0.001,
    gamma=0.06,
    epsilon=0.02,
    total_steps=7000,
    burn_in=5000,
    seed=seed,
    x0_a=-1.0,
    x0_b=-1.2,
    y0_a=-3.5,
    y0_b=-3.7,
)
observable_config = ObservableConfig(mode='identity_quadratic', polynomial_degree=2, include_bias=False)
workflow_config = WorkflowConfig(
    simulation=simulation_config,
    observables=observable_config,
    lag_steps=1,
    rank=2,
    alpha=1.0,
    ridge=1e-10,
    eps=1e-10,
    include_closed_form_ce=True,
    results_dir=results_dir,
)
config_df = pd.DataFrame([
    {
        'n_a': simulation_config.n_a,
        'n_b': simulation_config.n_b,
        'alpha_a': simulation_config.alpha_a,
        'alpha_b': simulation_config.alpha_b,
        'sigma_a': simulation_config.sigma_a,
        'sigma_b': simulation_config.sigma_b,
        'mu': simulation_config.mu,
        'gamma': simulation_config.gamma,
        'epsilon': simulation_config.epsilon,
        'T': simulation_config.total_steps,
        'transients': simulation_config.burn_in,
        'seed': simulation_config.seed,
        'x0_a': simulation_config.x0_a,
        'x0_b': simulation_config.x0_b,
        'y0_a': simulation_config.y0_a,
        'y0_b': simulation_config.y0_b,
    }
])
display(config_df)


## 3. ????

????????

- ? `example_maps_Q` ??????????? Rulkov ??
- ?????? `5000` ?????? `2000` ????????
- ??????????????


In [ ]:
data = generate_two_population_neuron_data(simulation_config)
display(Markdown(f"- reference available: `{data['reference_info']['available']}`"))
display(Markdown(f"- source note: {data['source_note']}"))
display(Markdown(f"- ? notebook ???????? Koopman ???????? `[{simulation_config.burn_in}, {simulation_config.total_steps})` ???????? `{data['state_matrix'].shape[0]}`?"))
summary_df = pd.DataFrame([data['summary']])
display(summary_df)
print('sync_state =', data['summary']['sync_state'])
print('state_matrix shape after burn-in =', data['state_matrix'].shape)


## 4. ??????

????????

- ?? `plot_neuron_analysis_combo(simulation_results[i])` ?????
- ????????????????????


In [ ]:
fig_micro, _ = plot_neuron_analysis_combo(data, time_window=None, cmap='vlag')
display(fig_micro)
plt.close(fig_micro)


## 5. ??????

????????

- ???????? `x/y` ??????????????
- ?????????????


In [ ]:
state_matrix, state_names = build_state_matrix_from_population_data(data)
state_preview = pd.DataFrame(state_matrix[:5, :12], columns=state_names[:12])
print('state_matrix shape =', state_matrix.shape)
print('number of state channels =', len(state_names))
display(state_preview)


## 6. ?? observables library

????????

- ?? `identity + quadratic` ?? lifting ???
- ?? lifting ???????????


In [ ]:
lifted_matrix, feature_names = build_observables(state_matrix, state_names, workflow_config.observables)
library_preview = pd.DataFrame({
    'feature_name': feature_names[:20],
    't0': lifted_matrix[0, :20],
    't1': lifted_matrix[1, :20],
})
print('lifted_matrix shape =', lifted_matrix.shape)
print('number of lifted features =', len(feature_names))
display(library_preview)


## 7. ????

????????

- ?? `0313`?? `pysindy` ?? lifting ???????????
- ?? `model.print` ????????`model.score` ???????
- ?????? `800`?????????????? 12 ???


In [ ]:
x_data_lift = lifted_matrix[:-1]
y_data_lift = lifted_matrix[1:]
identity_library = ps.IdentityLibrary()
optimizer = ps.STLSQ(threshold=0.0, alpha=1e-8, verbose=False)
model = ps.SINDy(feature_library=identity_library, optimizer=optimizer, discrete_time=discrete_time)
model.fit(lifted_matrix, t=dt, feature_names=feature_names)
score_value = model.score(lifted_matrix, t=dt)
A_raw = model.coefficients()
equations = model.equations()
nonzero_ratio = np.count_nonzero(A_raw) / A_raw.size
print('model.print ?????? 12 ????:')
for idx, equation in enumerate(equations[:12], start=1):
    print(f'{idx:02d}. {equation}')
print('...')
print(f'model.score = {score_value:.6f}')
print('A_raw shape =', A_raw.shape)
print(f'nonzero_ratio = {nonzero_ratio:.6f}')


## 8. ????????

????????

- ???????? `C00/C01/C11`
- ?????? K ????????


In [ ]:
koop_fit = fit_data_koopman_operator(
    [lifted_matrix],
    weights='uniform',
    eps=workflow_config.eps,
    ridge=workflow_config.ridge,
    lag_steps=workflow_config.lag_steps,
)
C00 = koop_fit['C00']
C01 = koop_fit['C01']
C11 = koop_fit['C11']
C00_inv_sqrt = koop_fit['C00_inv_sqrt']
C11_inv_sqrt = koop_fit['C11_inv_sqrt']
preview_n = 10
display(pd.DataFrame(C00[:preview_n, :preview_n], index=feature_names[:preview_n], columns=feature_names[:preview_n]))
display(pd.DataFrame(C01[:preview_n, :preview_n], index=feature_names[:preview_n], columns=feature_names[:preview_n]))
display(pd.DataFrame(C11[:preview_n, :preview_n], index=feature_names[:preview_n], columns=feature_names[:preview_n]))


## 9. ????????

????????

- ?????? `800 x 800` ????????
- ????????????????????


In [ ]:
fig_c00, _ = plot_matrix_heatmap(C00, feature_names, 'C00 covariance', figsize=(11, 10), label_step=100)
fig_c01, _ = plot_matrix_heatmap(C01, feature_names, 'C01 covariance', figsize=(11, 10), label_step=100)
fig_c11, _ = plot_matrix_heatmap(C11, feature_names, 'C11 covariance', figsize=(11, 10), label_step=100)
display(fig_c00)
display(fig_c01)
display(fig_c11)
plt.close(fig_c00)
plt.close(fig_c01)
plt.close(fig_c11)


## 10. ?? K ???? SVD ??

????????

- ? `0313` ?????????????
- ???????`A_step_data`?`K_bar_model`?`K_bar`
- ???????????????


In [ ]:
xdot_model = model.predict(x_data_lift)
err_no_t = np.linalg.norm(xdot_model - x_data_lift @ A_raw.T) / np.linalg.norm(xdot_model)
err_t = np.linalg.norm(xdot_model - x_data_lift @ A_raw) / np.linalg.norm(xdot_model)
A_ct = A_raw if err_t <= err_no_t else A_raw.T
print(f'orientation check: err(A)={err_t:.3e}, err(A.T)={err_no_t:.3e}')

A_step_model = scipy.linalg.expm(A_ct * dt) if not discrete_time else A_ct
A_step_data = koop_fit['A']
K_bar = koop_fit['K_bar']
model_whitening = whiten_operator_matrix(A_step_model, C00, C11, eps=workflow_config.eps)
K_bar_model = model_whitening['A_bar']

print(f'||A_step_data - A_step_model||_F = {np.linalg.norm(A_step_data - A_step_model):.3e}')
print(f'max sv(A_step_data)      = {np.linalg.svd(A_step_data, compute_uv=False)[0]:.6f}')
print(f'max sv(K_bar_model)     = {np.linalg.svd(K_bar_model, compute_uv=False)[0]:.6f}')
print(f'max sv(K_bar empirical) = {np.linalg.svd(K_bar, compute_uv=False)[0]:.6f}')


## 11. ?? K ????

????????

- ?? `Data-fitted step operator`
- ?? `Model operator with correct whitening`
- ?? `Empirical whitened Koopman`


In [ ]:
fig_k1, _ = plot_matrix_heatmap(A_step_data, feature_names, 'Data-fitted step operator', figsize=(11, 10), label_step=100)
fig_k2, _ = plot_matrix_heatmap(K_bar_model, feature_names, 'Model operator with correct whitening', figsize=(11, 10), label_step=100)
fig_k3, _ = plot_matrix_heatmap(K_bar, feature_names, 'Empirical whitened Koopman', figsize=(11, 10), label_step=100)
display(pd.DataFrame(A_step_data[:8, :8], index=feature_names[:8], columns=feature_names[:8]))
display(pd.DataFrame(K_bar_model[:8, :8], index=feature_names[:8], columns=feature_names[:8]))
display(pd.DataFrame(K_bar[:8, :8], index=feature_names[:8], columns=feature_names[:8]))
display(fig_k1)
display(fig_k2)
display(fig_k3)
plt.close(fig_k1)
plt.close(fig_k2)
plt.close(fig_k3)


## 12. ????

????????

- ?? `0313`?????????????
- ???? `??? a1`??? `10` ?
- ???? `model operator` ? `data-fitted step operator` ?????


In [ ]:
start_idx = 100
horizon = 10
selected_x = feature_names.index('x_a1')
selected_y = feature_names.index('y_a1')
true_segment = lifted_matrix[start_idx:start_idx + horizon + 1]

model_roll = np.zeros_like(true_segment)
data_roll = np.zeros_like(true_segment)
model_roll[0] = lifted_matrix[start_idx]
data_roll[0] = lifted_matrix[start_idx]
for step in range(horizon):
    model_roll[step + 1] = model.predict(model_roll[[step]])[0]
    data_roll[step + 1] = data_roll[step] @ A_step_data

prediction_df = pd.DataFrame({
    'step': np.arange(horizon + 1),
    'true_x_a1': true_segment[:, selected_x],
    'model_x_a1': model_roll[:, selected_x],
    'data_x_a1': data_roll[:, selected_x],
    'true_y_a1': true_segment[:, selected_y],
    'model_y_a1': model_roll[:, selected_y],
    'data_y_a1': data_roll[:, selected_y],
})
fig_pred, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(prediction_df['step'], prediction_df['true_x_a1'], 'o-', label='true x_a1')
axes[0].plot(prediction_df['step'], prediction_df['model_x_a1'], 's--', label='model x_a1')
axes[0].plot(prediction_df['step'], prediction_df['data_x_a1'], '^--', label='data-fitted x_a1')
axes[0].set_ylabel('x_a1')
axes[0].legend()
axes[0].grid(True, alpha=0.25)
axes[1].plot(prediction_df['step'], prediction_df['true_y_a1'], 'o-', label='true y_a1')
axes[1].plot(prediction_df['step'], prediction_df['model_y_a1'], 's--', label='model y_a1')
axes[1].plot(prediction_df['step'], prediction_df['data_y_a1'], '^--', label='data-fitted y_a1')
axes[1].set_ylabel('y_a1')
axes[1].set_xlabel('prediction step')
axes[1].legend()
axes[1].grid(True, alpha=0.25)
fig_pred.suptitle('10-step prediction for neuron a1', y=0.995)
fig_pred.tight_layout()
display(prediction_df)
display(fig_pred)
plt.close(fig_pred)


## 13. ????

????????

- ??? K ??? SVD
- ?????????? 15 ?????
- `model + correct whitening` ??? marker???? empirical ?????????


In [ ]:
raw_operator_singular_values = np.linalg.svd(A_step_data, compute_uv=False)
model_whitened_singular_values = np.linalg.svd(K_bar_model, compute_uv=False)
singular_values = np.linalg.svd(K_bar, compute_uv=False)
sv_fig_all, _ = plot_singular_value_comparison(raw_operator_singular_values, model_whitened_singular_values, singular_values)
sv_fig_top15, _ = plot_singular_value_comparison(raw_operator_singular_values, model_whitened_singular_values, singular_values, top_n=15)
sv_df = pd.DataFrame({
    'index': np.arange(1, min(20, len(singular_values)) + 1),
    'data_fitted_step': raw_operator_singular_values[:20],
    'model_correct_whitening': model_whitened_singular_values[:20],
    'empirical_whitened_koopman': singular_values[:20],
})
display(sv_df)
display(sv_fig_all)
display(sv_fig_top15)
plt.close(sv_fig_all)
plt.close(sv_fig_top15)


## 14. ????????

????????

- ??????? `positive contributions`
- ?? `0313` ???????


In [ ]:
positive_contributions = get_positive_contributions(singular_values.tolist())
positive_df = pd.DataFrame({
    'dimension': np.arange(1, len(positive_contributions) + 1),
    'positive_contribution': positive_contributions,
})
fig_pc, _ = plot_positive_contributions(positive_contributions)
display(positive_df.head(20))
display(fig_pc)
plt.close(fig_pc)


## 15. SVD ???rank ???EC ? CE

????????

- ? empirical whitened Koopman ? SVD
- ? rank ??????????
- ?? EC?CE ??? CE


In [ ]:
u, s, vt = np.linalg.svd(K_bar, full_matrices=False)
rank = workflow_config.rank
u_r = u[:, :rank]
s_r = s[:rank]
vt_r = vt[:rank, :]
coarse_matrix = C00_inv_sqrt @ u_r
macro_names = [f'z{i + 1}' for i in range(rank)]
macro_series = lifted_matrix @ coarse_matrix
macro_operator = np.linalg.pinv(macro_series[:-1]) @ macro_series[1:]

ce_channel_scores = channel_scores_from_singular_values(s, alpha=workflow_config.alpha, rank=rank)
ce_score = float(np.sum(ce_channel_scores))
ce_total_from_kbar = float(koopman_ce_total_score_from_kbar(K_bar, alpha=workflow_config.alpha, eps=workflow_config.eps))
ec_increments = get_positive_contributions(s[:rank].tolist())
ec_score = float(compute_entropy(ec_increments))
residual_covariance = compute_residual_covariance(x_data_lift, y_data_lift, A_step_model, eps=workflow_config.eps)
closed_form_ce = None
if workflow_config.include_closed_form_ce:
    closed_form_ce = float(liu2025_log_gamma_gis(A_step_model, residual_covariance, alpha=workflow_config.alpha, eps=workflow_config.eps))
metrics_df = pd.DataFrame([
    {
        'rank': rank,
        'EC': ec_score,
        'CE_rank_sum': ce_score,
        'CE_total_from_kbar': ce_total_from_kbar,
        'closed_form_ce': closed_form_ce,
    }
])
display(metrics_df)
print('top singular values =', s[:10])
print('EC increments =', ec_increments)
print('CE channel scores =', ce_channel_scores)


## 16. ???????????

????????

- ?????????????????????????????
- ????????????????


In [ ]:
left_sv_fig, _ = plot_rectangular_heatmap(u_r, feature_names, macro_names, 'Left singular vectors (full)', figsize=(7, 12), show_row_labels=False)
coarse_fig, _ = plot_rectangular_heatmap(coarse_matrix, feature_names, macro_names, 'Coarse-graining matrix', figsize=(7, 12), show_row_labels=False)
coarse_equations = format_equations(coarse_matrix, feature_names, macro_names, threshold=0.08)
display(pd.DataFrame(u_r[:12, :], index=feature_names[:12], columns=macro_names))
display(pd.DataFrame(coarse_matrix[:12, :], index=feature_names[:12], columns=macro_names))
for equation in coarse_equations[:rank]:
    print(equation)
display(left_sv_fig)
display(coarse_fig)
plt.close(left_sv_fig)
plt.close(coarse_fig)


## 17. ?????????/????

????????

- ????????
- ????????? vs ?????????
- ?????????????


In [ ]:
macro_fig, _ = plot_macro_series(macro_series, macro_names, max_points=500)
picked_micro_indices = [0, 1, 2 * simulation_config.n_a, 2 * simulation_config.n_a + 1]
comparison_fig, _ = plot_micro_macro_comparison(state_matrix, state_names, macro_series, macro_names, picked_indices=picked_micro_indices, max_points=500)
display(pd.DataFrame(macro_series[:10, :], columns=macro_names))
display(macro_fig)
display(comparison_fig)
plt.close(macro_fig)
plt.close(comparison_fig)


## 18. ???????

????????

- ????????????????
- ????????????


In [ ]:
macro_equations = format_equations(macro_operator, macro_names, [f'{name}_next' for name in macro_names], threshold=1e-4)
display(pd.DataFrame(macro_operator, index=macro_names, columns=macro_names))
for equation in macro_equations:
    print(equation)


## 19. ????

????????

- ???? csv/json/png ? `exp/map/results/`
- ?????????????????????


In [ ]:
prediction_df.to_csv(results_dir / 'multistep_prediction_neuron_a1.csv', index=False)
fig_save_pred, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(prediction_df['step'], prediction_df['true_x_a1'], 'o-', label='true x_a1')
axes[0].plot(prediction_df['step'], prediction_df['model_x_a1'], 's--', label='model x_a1')
axes[0].plot(prediction_df['step'], prediction_df['data_x_a1'], '^--', label='data-fitted x_a1')
axes[0].legend()
axes[0].grid(True, alpha=0.25)
axes[1].plot(prediction_df['step'], prediction_df['true_y_a1'], 'o-', label='true y_a1')
axes[1].plot(prediction_df['step'], prediction_df['model_y_a1'], 's--', label='model y_a1')
axes[1].plot(prediction_df['step'], prediction_df['data_y_a1'], '^--', label='data-fitted y_a1')
axes[1].legend()
axes[1].grid(True, alpha=0.25)
axes[1].set_xlabel('prediction step')
fig_save_pred.suptitle('10-step prediction for neuron a1', y=0.995)
fig_save_pred.tight_layout()
fig_save_pred.savefig(results_dir / 'multistep_prediction_neuron_a1.png', bbox_inches='tight')
plt.close(fig_save_pred)

summary_metrics = pd.DataFrame([
    {
        'rank': rank,
        'n_states': state_matrix.shape[1],
        'n_features': lifted_matrix.shape[1],
        'model_score': score_value,
        'EC': ec_score,
        'CE_rank_sum': ce_score,
        'CE_total_from_kbar': ce_total_from_kbar,
        'closed_form_ce': closed_form_ce,
    }
])
workflow = {
    'config': workflow_config,
    'simulation': data,
    'state_matrix': state_matrix,
    'state_names': state_names,
    'lifted_matrix': lifted_matrix,
    'feature_names': feature_names,
    'koopman': koop_fit,
    'operator_matrix': A_step_data,
    'model_whitened_matrix': K_bar_model,
    'whitened_matrix': K_bar,
    'raw_operator_singular_values': raw_operator_singular_values,
    'model_whitened_singular_values': model_whitened_singular_values,
    'singular_values': s,
    'singular_vectors_left': u,
    'rank': rank,
    'macro_names': macro_names,
    'coarse_matrix': coarse_matrix,
    'macro_series': macro_series,
    'macro_operator': macro_operator,
    'summary': summary_metrics,
    'ec_increments': ec_increments,
    'ce_channel_scores': ce_channel_scores,
    'ec_score': ec_score,
    'ce_score': ce_score,
    'ce_total_from_kbar': ce_total_from_kbar,
    'closed_form_ce': closed_form_ce,
    'coarse_equations': coarse_equations,
    'macro_equations': macro_equations,
    'positive_contributions': positive_contributions,
}
artifacts = save_workflow_results(workflow, results_dir)
display(summary_metrics)
print('saved files =', len(artifacts))
for key in sorted(list(artifacts))[:16]:
    print(key, '->', artifacts[key])
